# Chapter 13 &mdash; The Formal Turing Machine $(Q,\Sigma,\Gamma,\Delta,q_0,B,F)$

**Concept 6 of the Chapter 13 decomposition:** *The Formal Turing Machine $(Q,\Sigma,\Gamma,\Delta,q_0,B,F)$*

$\Delta$ maps (state, tape symbol) to (state, symbol, $L/R/S$) &mdash; a <i>set</i> of them if nondeterministic.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Formal-TM/Concept-Formal-TM.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateTM as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateTM, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


$$T = (Q,\Sigma,\Gamma,\Delta,q_0,B,F)$$

* $Q$ &mdash; finite states;
* $\Sigma$ &mdash; **input** alphabet;
* $\Gamma \supseteq \Sigma$ &mdash; **tape** alphabet, which also contains the blank;
* $B \in \Gamma\setminus\Sigma$ &mdash; the **blank**, written `.` in Jove;
* $\Delta: Q\times\Gamma \to \mathcal{P}(Q\times\Gamma\times\{L,R,S\})$;
* $q_0$, $F$ &mdash; start and final states.

Two differences from a PDA. The tape alphabet **includes** the input alphabet (you
write onto the same tape you read), and each move specifies a **direction** as well as
a symbol.

Jove's syntax: `State : read ; write , dir -> State`, with `.` for the blank and
`|` for alternatives.

## 2. Definitions

### A machine and its seven components

In [ ]:
T = md2mc('''TM
!! replace every 0 by X, leave 1s alone, then halt
I : 0 ; X , R -> I
I : 1 ; 1 , R -> I
I : . ; . , S -> F
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

### Reading the components off

In [ ]:
def show_tm(T):
    for k in ['Q', 'Sigma', 'Gamma', 'q0', 'B', 'F']:
        v = T[k]
        print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))
    print("Delta :")
    for (q, r), outs in sorted(T["Delta"].items()):
        for (q2, w, d) in sorted(outs):
            print("   (%s, %s) -> (%s, %s, %s)" % (q, r, q2, w, d))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;5.&nbsp;The Long List of Universal Computers, and Tape Simulation by Two Stacks](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Universal-Computers-And-Two-Stacks/Concept-Universal-Computers-And-Two-Stacks.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;7.&nbsp;Halting, Acceptance, and Why a TM Need Not Read Its Input](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Halting-And-Acceptance/Concept-Halting-And-Acceptance.ipynb)&nbsp;&rarr;

---

## 3. Tests

All seven.

In [ ]:
show_tm(T)

$\Gamma$ **contains** $\Sigma$, plus the blank and any working symbols.

In [ ]:
print("Sigma        :", sorted(T["Sigma"]))
print("Gamma        :", sorted(T["Gamma"]))
print("blank        :", T["B"])
print("Gamma - Sigma:", sorted(T["Gamma"] - T["Sigma"]))
assert T["Sigma"] <= T["Gamma"]
assert T["B"] in T["Gamma"] and T["B"] not in T["Sigma"]
print("\n'X' is a WORKING symbol: written on the tape, never read from the input.")

Each move writes **and** moves.

In [ ]:
for (q, r), outs in sorted(T["Delta"].items()):
    for (q2, w, d) in sorted(outs):
        assert d in ('L', 'R', 'S')
print("every transition names a direction in {L, R, S}")

And the machine does what it says.

In [ ]:
for t in ['0101', '111', '000']:
    out = tm_tape(T, t, fuel=60)[0]
    print("  %-7r -> %r" % (t, out))
    assert out == t.replace('0', 'X')

A **nondeterministic** TM: $\Delta$ returns more than one triple.

In [ ]:
N = md2mc('''TM
I : 0 ; 0 , R -> I
I : 0 ; 1 , R -> F
''')
multi = [(k, sorted(v)) for k, v in N["Delta"].items() if len(v) > 1]
print("nondeterministic entries :", multi)
assert multi
print("accepts '00' ?", tm_accepts(N, '00'))
assert tm_accepts(N, '00')

## 4. Animation

The 0-to-X rewriter.

In [ ]:
from jove.AnimateTM import *
AnimateTM(T, FuseEdges=True)

## 5. Exercises


1. Why must the blank be **outside** $\Sigma$?
2. What is the smallest useful $\Gamma$? Can you always get down to $\{0,1,B\}$?
3. Write a TM whose $\Gamma$ has two working symbols. What are they for?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter13/Concept-Formal-TM')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')